In [103]:
import warnings
warnings.filterwarnings("ignore")

In [104]:
from ucimlrepo import fetch_ucirepo 
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

from aif360.sklearn.metrics import consistency_score
from fairlearn.postprocessing import ThresholdOptimizer
from fairlearn.metrics import demographic_parity_difference, demographic_parity_ratio
from sklearn.metrics import accuracy_score

## German

### Preprocessing

In [105]:
from ucimlrepo import fetch_ucirepo 
  
# fetch dataset 
statlog_german_credit_data = fetch_ucirepo(id=144) 
  
# data (as pandas dataframes) 
X = statlog_german_credit_data.data.features 
y = statlog_german_credit_data.data.targets 

In [106]:
X = pd.get_dummies(X, drop_first = True)
for_work = X['Attribute20_A202']

In [107]:
y = y-1 #change to zeros and ones

### Original Scores

Demographic Parity Difference

In [108]:
dp_diff = demographic_parity_difference(y, y, sensitive_features=for_work)
print(dp_diff)

0.19926468524599367


Disparate Impact Ratio (DIR)/Demographic Parity Ratio in Fairlearn

In [109]:
dir_score = demographic_parity_ratio(y, y, sensitive_features=for_work)
print(dir_score)

0.3517165814463112


Consistency Score

In [110]:
cons_scaler  = StandardScaler()

x_cons = cons_scaler.fit_transform(X)
cons_score = consistency_score(x_cons, y, n_neighbors=5)
cons_score

np.float64(0.729)

### Train-Test Split

In [111]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, stratify=y, random_state=123
)

In [112]:
for_train = X_train['Attribute20_A202']
for_test = X_test['Attribute20_A202']

### Scaling Model

In [113]:
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

### Fitting Model

In [114]:
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",1000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary <random_state>` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is '

In [115]:
train_preds = model.predict(X_train)
test_preds  = model.predict(X_test)

### Bias Mitigation: Threshold Optimizer

In [116]:
threshold_optimizer = ThresholdOptimizer(
    estimator=model,
    constraints="demographic_parity",
    objective="balanced_accuracy_score",
    predict_method="predict_proba",
    prefit=True,
)

threshold_optimizer.fit(X_train, y_train, sensitive_features=for_train)

,estimator,LogisticRegre...max_iter=1000)
,objective,'balanced_accuracy_score'
,prefit,True
,predict_method,'predict_proba'
,constraints,'demographic_parity'
,grid_size,1000
,flip,False
,tol,None
Name,Type,Value
estimator_,LogisticRegression,LogisticRegre...max_iter=1000)
interpolated_thresholder_,InterpolatedThresholder,"InterpolatedT..., prefit=True)"


In [117]:
y_pred_post_to = threshold_optimizer.predict(X_test, sensitive_features=for_test, random_state=123)

In [118]:
dp_diff_test_pre = demographic_parity_difference(y_test, test_preds, sensitive_features=for_test)
dp_diff_test_post = demographic_parity_difference(y_test, y_pred_post_to, sensitive_features=for_test)

In [119]:
dir_score_test_pre = demographic_parity_ratio(y_test, test_preds, sensitive_features=for_test)
dir_score_test_post = demographic_parity_ratio(y_test, y_pred_post_to, sensitive_features=for_test)

In [120]:
cons_pre = consistency_score(X_test, test_preds, n_neighbors=5)
cons_post  = consistency_score(X_test, y_pred_post_to, n_neighbors=5)

In [121]:
acc_pre = accuracy_score(y_test, test_preds)
acc_post = accuracy_score(y_test, y_pred_post_to)

### Score Comparison Pre vs Post Mitigation

In [122]:
score_summary = pd.DataFrame(
    [[dp_diff_test_pre, dp_diff_test_post],
     [dir_score_test_pre, dir_score_test_post],
     [cons_pre, cons_post],
     [acc_pre,acc_post]],
    index=["DP Diff", "DIR Score", "Consistency","Accuracy"],
    columns=["Pre", "Post"])

print(score_summary)

                  Pre      Post
DP Diff      0.273859  0.106501
DIR Score    0.000000  0.757862
Consistency  0.784800  0.711200
Accuracy     0.748000  0.712000


### Investigation

In [123]:
print(sum(for_work))

37


In [124]:
print(len(X))

1000


In [125]:
for_work_df = X[X['Attribute20_A202'] == True]
y_for_work_df = y.loc[for_work_df.index]
print(sum(y_for_work_df['class'])/len(y_for_work_df)*100,'% of Foreign Workers have bad credit risk')

print((sum(y['class'])/len(y))*100, '% of everyone has bad credit risk')

10.81081081081081 % of Foreign Workers have bad credit risk
30.0 % of everyone has bad credit risk


## Adult

### Preprocessing

In [126]:
adult = pd.read_csv("data/adult.csv")
adult = pd.DataFrame(adult)

In [127]:
adult['high_income'] = np.where(adult['income'] == '>50K', 1,0)
adult = adult.drop(["income"], axis=1)

In [128]:
x = adult.drop(["high_income"], axis=1)
x = pd.get_dummies(x, drop_first = True)
y = adult['high_income']
sex = x['sex_Male']

### Original Scores

Demographic Parity Difference

In [129]:
dp_diff = demographic_parity_difference(y, y, sensitive_features=sex)
print(dp_diff)

0.19627598779361352


Disparate Impact Ratio (DIR)/Demographic Parity Ratio in Fairlearn

In [130]:
dir_score = demographic_parity_ratio(y, y, sensitive_features=sex)
print(dir_score)

0.3580225496813511


Consistency Score

In [131]:
cons_scaler  = StandardScaler()

x_cons = cons_scaler.fit_transform(x)
cons_score = consistency_score(x_cons, y, n_neighbors=5)
cons_score

np.float64(0.8285126378182488)

### Train-Test Split

In [132]:
X_train, X_test, y_train, y_test = train_test_split(
    x, y, stratify=y, random_state=123
)

In [133]:
sex_train = X_train['sex_Male']
sex_test = X_test['sex_Male']

### Scaling Model

In [134]:
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

### Fitting Model

In [135]:
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",1000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary <random_state>` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is '

In [136]:
train_preds = model.predict(X_train)
test_preds  = model.predict(X_test)

### Bias Mitigation: Threshold Optimizer

In [137]:
threshold_optimizer = ThresholdOptimizer(
    estimator=model,
    constraints="demographic_parity",
    objective="balanced_accuracy_score",
    predict_method="predict_proba",
    prefit=True,
)

threshold_optimizer.fit(X_train, y_train, sensitive_features=sex_train)

,estimator,LogisticRegre...max_iter=1000)
,objective,'balanced_accuracy_score'
,prefit,True
,predict_method,'predict_proba'
,constraints,'demographic_parity'
,grid_size,1000
,flip,False
,tol,None
Name,Type,Value
estimator_,LogisticRegression,LogisticRegre...max_iter=1000)
interpolated_thresholder_,InterpolatedThresholder,"InterpolatedT..., prefit=True)"


In [138]:
y_pred_post_to = threshold_optimizer.predict(X_test, sensitive_features=sex_test, random_state=123)

In [139]:
dp_diff_test_pre = demographic_parity_difference(y_test, test_preds, sensitive_features=sex_test)
dp_diff_test_post = demographic_parity_difference(y_test, y_pred_post_to, sensitive_features=sex_test)

In [140]:
dir_score_test_pre = demographic_parity_ratio(y_test, test_preds, sensitive_features=sex_test)
dir_score_test_post = demographic_parity_ratio(y_test, y_pred_post_to, sensitive_features=sex_test)

In [141]:
cons_pre = consistency_score(X_test, test_preds, n_neighbors=5)
cons_post  = consistency_score(X_test, y_pred_post_to, n_neighbors=5)

In [142]:
acc_pre = accuracy_score(y_test, test_preds)
acc_post = accuracy_score(y_test, y_pred_post_to)

### Score Comparison Pre vs Post Mitigation

In [143]:
score_summary = pd.DataFrame(
    [[dp_diff_test_pre, dp_diff_test_post],
     [dir_score_test_pre, dir_score_test_post],
     [cons_pre, cons_post],
     [acc_pre,acc_post]],
    index=["DP Diff", "DIR Score", "Consistency","Accuracy"],
    columns=["Pre", "Post"])

print(score_summary)

                  Pre      Post
DP Diff      0.169943  0.003511
DIR Score    0.324079  0.989015
Consistency  0.925660  0.902125
Accuracy     0.851861  0.801867


### Investigation

In [144]:
print(sum(sex))

21790


In [146]:
print(len(x))

32561


In [154]:
male_df = x[x['sex_Male'] == True]
y_male_df = y.loc[male_df.index]
print(sum(y_male_df)/len(y_male_df)*100,'% of men have high income')
print((sum(y)/len(y))*100, '% of everyone have high income')

30.573657641119777 % of men have high income
24.080955744602438 % of everyone have high income


## COMPAS

### Preprocessing

In [155]:
compas = pd.read_csv("data\propublica_data_for_fairml.csv")

In [156]:
y = compas['Two_yr_Recidivism']
X = compas.drop(['Two_yr_Recidivism'], axis=1)
af_am = X['African_American']

### Original Scores

Demographic Parity Difference

In [157]:
dp_diff = demographic_parity_difference(y, y, sensitive_features=af_am)
print(dp_diff)

0.14009988991616285


Disparate Impact Ratio (DIR)/Demographic Parity Ratio in Fairlearn

In [158]:
dir_score = demographic_parity_ratio(y, y, sensitive_features=af_am)
print(dir_score)

0.7321991869453238


Consistency Score

In [159]:
cons_scaler  = StandardScaler()

x_cons = cons_scaler.fit_transform(X)
cons_score = consistency_score(x_cons, y, n_neighbors=5)
cons_score

np.float64(0.6290667530784186)

### Train-Test Split

In [160]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, stratify=y, random_state=123
)

In [161]:
afam_train = X_train['African_American']
afam_test = X_test['African_American']

### Scaling Model

In [162]:
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

### Fitting Model

In [163]:
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",1000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary <random_state>` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is '

In [164]:
train_preds = model.predict(X_train)
test_preds  = model.predict(X_test)

### Bias Mitigation: Threshold Optimizer

In [165]:
threshold_optimizer = ThresholdOptimizer(
    estimator=model,
    constraints="demographic_parity",
    objective="balanced_accuracy_score",
    predict_method="predict_proba",
    prefit=True,
)

threshold_optimizer.fit(X_train, y_train, sensitive_features=afam_train)

,estimator,LogisticRegre...max_iter=1000)
,objective,'balanced_accuracy_score'
,prefit,True
,predict_method,'predict_proba'
,constraints,'demographic_parity'
,grid_size,1000
,flip,False
,tol,None
Name,Type,Value
estimator_,LogisticRegression,LogisticRegre...max_iter=1000)
interpolated_thresholder_,InterpolatedThresholder,"InterpolatedT..., prefit=True)"


In [166]:
y_pred_post_to = threshold_optimizer.predict(X_test, sensitive_features=afam_test, random_state=123)

In [167]:
dp_diff_test_pre = demographic_parity_difference(y_test, test_preds, sensitive_features=afam_test)
dp_diff_test_post = demographic_parity_difference(y_test, y_pred_post_to, sensitive_features=afam_test)

In [168]:
dir_score_test_pre = demographic_parity_ratio(y_test, test_preds, sensitive_features=afam_test)
dir_score_test_post = demographic_parity_ratio(y_test, y_pred_post_to, sensitive_features=afam_test)

In [169]:
cons_pre = consistency_score(X_test, test_preds, n_neighbors=5)
cons_post  = consistency_score(X_test, y_pred_post_to, n_neighbors=5)

In [170]:
acc_pre = accuracy_score(y_test, test_preds)
acc_post = accuracy_score(y_test, y_pred_post_to)

### Score Comparison Pre vs Post Mitigation

In [171]:
score_summary = pd.DataFrame(
    [[dp_diff_test_pre, dp_diff_test_post],
     [dir_score_test_pre, dir_score_test_post],
     [cons_pre, cons_post],
     [acc_pre,acc_post]],
    index=["DP Diff", "DIR Score", "Consistency","Accuracy"],
    columns=["Pre", "Post"])

print(score_summary)

                  Pre      Post
DP Diff      0.285353  0.007996
DIR Score    0.465297  0.982207
Consistency  0.970447  0.977058
Accuracy     0.675956  0.657161


### Investigation

In [175]:
print(sum(af_am))

3175


In [178]:
print(len(X))

6172


In [173]:
afam_df = X[X['African_American'] == True]
y_afam_df = y.loc[afam_df.index]
print(sum(y_afam_df)/len(y_afam_df)*100,'% of African Americans have recidivism within 2 years')
print((sum(y)/len(y))*100, '% of everyone have recidivism within 2 years') 

52.31496062992126 % of African Americans have recidivism within 2 years
45.51198963058976 % of everyone have recidivism within 2 years
